# 21장. BoW의 한계

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 13장, 15장, 20장 |
| 이 장의 결과물 | 리포트용 "BoW 한계 분석" 사례 표, 개선 방향 표 |
| 관련 학습 목표 | ④ BoW 방식이 문맥과 어순을 반영하지 못하는 한계를 구체적 사례와 함께 설명 |

**학습 목표**
- 어순, 동음이의어, 동의어, 부정, OOV, 희소성 문제를 **실제 TF-IDF 벡터와 코사인 유사도**로 보여 줄 수 있다.
- 각 한계가 BoW의 어떤 가정에서 나오는지 설명할 수 있다.
- 20장의 오분류 사례를 이 한계와 연결할 수 있다.
- 한계마다 어떤 개선 방법이 있는지, 그 방법이 무엇을 해결하고 무엇을 남기는지 말할 수 있다.

과제의 처음 질문은 *"단어의 출현 빈도만으로 문서의 의미를 파악할 수 있는가?"*였습니다. 지금까지는 "꽤 된다"를 확인했다면(검색 precision, 분류 정확도), 이 장에서는 **어디서 무너지는지**를 확인합니다.

> 이 장의 모든 비교는 미니 예제가 아니라 **실제 코퍼스로 학습한** 어휘와 IDF를 씁니다. 그래서 "학습 데이터에 없는 단어는 버려진다" 같은 현실의 문제까지 함께 보입니다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

en = load_corpus("en")
ko = load_corpus("ko")
PREP = {"en": en.attrs["preprocessor"], "ko": ko.attrs["preprocessor"]}
VEC = {"en": NumpyTfidf(min_df=MIN_DF["en"]).fit(en["tokens"].tolist()),
       "ko": NumpyTfidf(min_df=MIN_DF["ko"]).fit(ko["tokens"].tolist())}
X_ko = VEC["ko"].transform(ko["tokens"].tolist())      # 한국어 검색용 문서 행렬 (15장)
for lang, df in [("en", en), ("ko", ko)]:
    print(f"{lang}: 문서 {len(df):,}건, 어휘 {len(VEC[lang].vocabulary_):,}개")

en: 문서 4,646건, 어휘 13,387개
ko: 문서 7,000건, 어휘 6,332개


문장 두 개를 같은 전처리와 같은 TF-IDF(코퍼스로 학습한 어휘·IDF)로 벡터로 바꾸고 코사인 유사도를 계산하는 함수를 만듭니다. 15장의 검색과 똑같은 계산입니다. 결과는 나중에 리포트용 표로 모으기 위해 `RESULTS`에 쌓아 둡니다.

In [2]:
RESULTS = []


def pair(kind: str, lang: str, a: str, b: str) -> None:
    prep, vec = PREP[lang], VEC[lang]
    ta, tb = prep(a), prep(b)
    va, vb = vec.transform([ta])[0], vec.transform([tb])[0]
    cos = float(va @ vb) if va.any() and vb.any() else 0.0      # 정규화된 벡터라 내적 = 코사인
    same = va.any() and np.allclose(va, vb)
    oov = sorted({t for t in ta + tb if t not in vec.vocabulary_})
    print(f"[{kind}] ({lang})")
    print(f'  A: "{a}" → {ta}')
    print(f'  B: "{b}" → {tb}')
    print(f"  코사인 유사도 = {cos:.3f}" + ("  (벡터 완전히 동일)" if same else ""))
    if oov:
        print(f"  어휘 사전에 없어 버려진 단어: {oov}")
    RESULTS.append({"한계": kind, "언어": lang, "문장 A": a, "문장 B": b, "코사인": round(cos, 3)})

## 21.1 어순 무시

BoW는 문서를 **단어 주머니**로 봅니다. 주머니에 무엇이 몇 개 들었는지만 기록하고 **순서는 버립니다**. 그래서 단어 구성이 같으면 뜻이 정반대여도 같은 벡터가 됩니다.

In [3]:
pair("어순 무시", "en", "the dog bit the man", "the man bit the dog")
print()
pair("어순 무시", "ko", "한국이 일본을 이겼다", "일본이 한국을 이겼다")

[어순 무시] (en)
  A: "the dog bit the man" → ['dog', 'bit', 'man']
  B: "the man bit the dog" → ['man', 'bit', 'dog']
  코사인 유사도 = 1.000  (벡터 완전히 동일)

[어순 무시] (ko)
  A: "한국이 일본을 이겼다" → ['한국', '일본', '이기다']
  B: "일본이 한국을 이겼다" → ['일본', '한국', '이기다']
  코사인 유사도 = 1.000  (벡터 완전히 동일)


두 쌍 모두 **벡터가 완전히 같습니다**. 누가 누구를 물었는지, 누가 이겼는지 정보가 사라졌습니다. 한국어는 형태소 분석이 조사(`이`, `을`)를 지웠기 때문에, 원래 문장에서 승자를 알려 주던 정보까지 없어졌습니다.

### 공백 토큰화는 어순을 볼까?
공백 토큰화에서는 `고양이가`와 `고양이를`이 다른 토큰입니다. 그러면 "고양이가 쥐를 잡았다"와 "쥐가 고양이를 잡았다"는 다른 벡터가 됩니다. 하지만 이것은 **조사(격)가 남아서** 생긴 차이이지, 어순을 본 것이 아닙니다. 순서만 바꾼 문장으로 확인합니다. `고양이`는 뉴스 제목 코퍼스에 거의 없어서, 여기서는 문장 네 개로 작은 TF-IDF를 학습해 비교합니다.

In [4]:
sentences = ["고양이가 쥐를 잡았다", "쥐가 고양이를 잡았다", "쥐를 고양이가 잡았다"]
for tok in ["whitespace", "okt"]:
    prep = KoreanPreprocessor(tok)
    tokens = [prep(s) for s in sentences]
    X = NumpyTfidf().fit_transform(tokens)
    print(f"[{tok}]")
    for s, t in zip(sentences, tokens):
        print(f"  {s:<14} → {t}")
    print(f"  cos(1번, 2번: 역할이 바뀜)     = {X[0] @ X[1]:.3f}")
    print(f"  cos(1번, 3번: 순서만 바뀜)     = {X[0] @ X[2]:.3f}\n")

[whitespace]
  고양이가 쥐를 잡았다    → ['고양이가', '쥐를', '잡았다']
  쥐가 고양이를 잡았다    → ['쥐가', '고양이를', '잡았다']
  쥐를 고양이가 잡았다    → ['쥐를', '고양이가', '잡았다']
  cos(1번, 2번: 역할이 바뀜)     = 0.185
  cos(1번, 3번: 순서만 바뀜)     = 1.000

[okt]
  고양이가 쥐를 잡았다    → ['고양이', '쥐', '잡다']
  쥐가 고양이를 잡았다    → ['쥐', '고양이', '잡다']
  쥐를 고양이가 잡았다    → ['쥐', '고양이', '잡다']
  cos(1번, 2번: 역할이 바뀜)     = 1.000
  cos(1번, 3번: 순서만 바뀜)     = 1.000



- **공백 토큰화**: 1번과 2번은 조사가 달라서 유사도가 낮지만, 순서만 바꾼 1번과 3번은 **1.000**입니다. 조사가 역할을 알려 줬을 뿐 어순은 여전히 보지 않습니다.
- **Okt**: 조사를 지웠으므로 세 문장이 모두 같은 벡터입니다.

## 21.2 동음이의어

글자가 같지만 뜻이 다른 단어(동음이의어, 다의어)는 BoW에서 **같은 차원**입니다. 문맥을 보지 않으므로 뜻을 구별할 수 없습니다.

In [5]:
pair("동음이의어", "ko", "배가 고프다", "배를 타고 바다로")
print()
pair("동음이의어", "en", "apple released a new mac", "apple pie recipe")

[동음이의어] (ko)
  A: "배가 고프다" → ['배', '고프다']
  B: "배를 타고 바다로" → ['배', '타고', '바다로']
  코사인 유사도 = 0.317
  어휘 사전에 없어 버려진 단어: ['타고']

[동음이의어] (en)
  A: "apple released a new mac" → ['apple', 'released', 'new', 'mac']
  B: "apple pie recipe" → ['apple', 'pie', 'recipe']
  코사인 유사도 = 0.385
  어휘 사전에 없어 버려진 단어: ['recipe']


신체의 배와 탈것의 배, 회사 애플과 과일 사과가 같은 차원이라서, 뜻이 전혀 다른 문장끼리 0보다 큰 유사도를 가집니다.

### 실제 코퍼스에서: `space`는 우주만 뜻할까?
영어 데이터에서 `space`는 `sci.space`의 핵심어입니다. 그런데 컴퓨터 게시판에도 `space`가 나옵니다. 어떤 뜻으로 쓰였는지 봅니다.

In [6]:
has_space = en["tokens"].map(lambda t: "space" in t)
print("'space'가 나온 문서 수 (카테고리별):")
print(en.loc[has_space, "label"].value_counts().to_string())

CONTEXT = re.compile(r"(?:\S+\s+){0,4}\bspace\b(?:\s+\S+){0,4}", re.IGNORECASE)
print("\ncomp.* 게시판에서 'space'가 쓰인 문맥:")
seen = []
for text in en.loc[has_space & en["label"].str.startswith("comp"), "text"]:
    m = CONTEXT.search(text)
    ctx = " ".join(m.group().split()) if m else ""
    if ctx and "@" not in ctx and "<" not in ctx and ctx not in seen:
        seen.append(ctx)
        print("  …" + ctx + "…")
    if len(seen) == 6:
        break

'space'가 나온 문서 수 (카테고리별):
label
sci.space             332
comp.graphics          40
comp.windows.x         34
rec.sport.hockey        3
rec.sport.baseball      3

comp.* 게시판에서 'space'가 쓰인 문맥:
  …Faster CPU's, More disk space…
  …to fill up disk space…
  …to 128M of disk space used for swapping *…
  …space intensive way possible. (i.e.…
  …of earth shot from space…
  …X excluding whitespace (tab, space…


컴퓨터 게시판의 `space`는 디스크·메모리 **공간**, 공백 문자(tab, space) 같은 뜻이 대부분이고, 우주에서 찍은 지구 사진처럼 실제 우주를 말하는 경우도 섞여 있습니다. TF-IDF는 이 모두를 **우주의 space와 같은 열**에 더합니다. 분류기가 `space`에 큰 `sci.space` 가중치를 배웠다면, "disk space" 질문이 우주 게시판 쪽으로 끌려갈 수 있습니다.

한국어 제목에서 `배`도 확인합니다.

In [7]:
has_bae = ko["tokens"].map(lambda t: "배" in t)
print("'배' 토큰이 있는 제목 수:", int(has_bae.sum()))
print(ko.loc[has_bae, ["text", "label"]].head(6).to_string(index=False))

q = VEC["ko"].transform([PREP["ko"]("배를 타고 바다로")])[0]
s = X_ko @ q
print('\nQuery: "배를 타고 바다로" 상위 5개')
for rank, i in enumerate(np.argsort(-s, kind="stable")[:5], start=1):
    print(f"  {rank}. [{s[i]:.3f}] {ko.loc[i, 'text']} ({ko.loc[i, 'label']})")

'배' 토큰이 있는 제목 수: 53
                             text label
셀트리온 작년 영업익 5천174억 사상 최대…1년새 2배종합    경제
            출력 20배 높인 마찰전기 발전기 개발  IT과학
노르웨이국부펀드 한국 등 신흥시장 채권 제외…환경투자 2배로    세계
 작년 인도네시아서 지진 1만1천여건 발생…예년의 2배 육박    세계
KIST·건국대 AI 바이러스 검출 키트 개발…민감도 10배  IT과학
삼성 갤노트10 국내 사전판매량 130만대…전작 대비 2배↑    경제

Query: "배를 타고 바다로" 상위 5개
  1. [0.351] 비오고 쌀쌀해도…산으로 들로 바다로 봄나들이 인산인해 (생활문화)
  2. [0.326] 부산국제영화제 오늘 화려한 개막…열흘간 영화의 바다로 (생활문화)
  3. [0.324] 만화의 바다로 풍덩…제20회 부천국제만화축제 개막 (생활문화)
  4. [0.287] 차세대 통신 5G 전송량 LTE의 5배 속도는 4050배 빨라 (IT과학)
  5. [0.266] 2022년까지 기초과학 연구비 2조5천억으로 2배 늘린다 (IT과학)


이 코퍼스의 `배`는 대부분 "2배", "10배"처럼 **곱절**이라는 뜻입니다. 그래서 선박 이야기인 "배를 타고 바다로"로 검색하면, 상위 1~3위는 `바다로`가 겹친 나들이·축제 기사이고, 4~5위는 선박과 전혀 상관없는 "N배 빨라", "2배 늘린다" 기사입니다. 쿼리의 `배`와 문서의 `배`가 같은 열이기 때문입니다. (`바다로`가 `바다`+`로`로 나뉘지 않고 한 토큰으로 남은 것은 9장에서 본 형태소 분석의 한계입니다.)

## 21.3 동의어와 표기 차이

반대로 **뜻이 같은데 글자가 다른** 단어는 서로 다른 차원입니다. 겹치는 차원이 없으면 내적이 0이므로 유사도도 0입니다.

In [8]:
pair("동의어·표기 차이", "ko", "북한", "北")
print()
pair("동의어·표기 차이", "ko", "자동차 판매", "차량 판매")
print()
pair("동의어·표기 차이", "en", "car", "automobile")
print()
pair("동의어·표기 차이", "en", "nasa shuttle launch", "space agency orbiter liftoff")

[동의어·표기 차이] (ko)
  A: "북한" → ['북한']
  B: "北" → ['北']
  코사인 유사도 = 0.000

[동의어·표기 차이] (ko)
  A: "자동차 판매" → ['자동차', '판매']
  B: "차량 판매" → ['차량', '판매']
  코사인 유사도 = 0.396

[동의어·표기 차이] (en)
  A: "car" → ['car']
  B: "automobile" → ['automobile']
  코사인 유사도 = 0.000
  어휘 사전에 없어 버려진 단어: ['automobile']

[동의어·표기 차이] (en)
  A: "nasa shuttle launch" → ['nasa', 'shuttle', 'launch']
  B: "space agency orbiter liftoff" → ['space', 'agency', 'orbiter', 'liftoff']
  코사인 유사도 = 0.000


- `북한`과 `北`, `nasa shuttle launch`와 `space agency orbiter liftoff`는 같은 대상·사건인데 유사도가 **0**입니다.
- "자동차 판매"와 "차량 판매"는 `판매`가 겹쳐서 0은 아니지만, 핵심어인 `자동차`/`차량`은 서로 기여하지 못합니다.

### 검색에서 어떤 일이 생기나
"북한"으로 검색하면 `北`만 쓴 제목은 아무리 북한 기사여도 점수가 0이라 **절대 찾을 수 없습니다.** 반대로 형태소 분석이 `북한산`을 `북한`+`산`으로 자르면, 산 이야기가 북한 검색에 끼어듭니다.

In [9]:
q = VEC["ko"].transform([PREP["ko"]("북한")])[0]
scores = X_ko @ q

print('Query: "북한" 상위 5개')
for rank, i in enumerate(np.argsort(-scores, kind="stable")[:5], start=1):
    print(f"  {rank}. [{scores[i]:.3f}] {ko.loc[i, 'text']} ({ko.loc[i, 'label']})")

only_hanja = ko["tokens"].map(lambda t: "北" in t and "북한" not in t)
print(f"\n'北'만 있고 '북한'은 없는 제목: {int(only_hanja.sum())}건 → 이 문서들의 최고 점수: {scores[only_hanja.to_numpy()].max():.3f}")
print("예:", ko.loc[only_hanja, "text"].head(3).tolist())

Query: "북한" 상위 5개
  1. [0.493] 북한 목선 인양준비 작업 (정치)
  2. [0.443] 북한 김정일 생일 경축 불꽃놀이 (정치)
  3. [0.409] 설산으로 변한 북한산은 한 폭의 수채화 (생활문화)
  4. [0.401] 북한에 허가받은 상설시장 450개…평남 최다 (정치)
  5. [0.391] 북한 김정은 낮 12시 30분 신년사 발표 (정치)

'北'만 있고 '북한'은 없는 제목: 171건 → 이 문서들의 최고 점수: 0.000
예: ['北 고려항공 中 칭다오 운항…관광객 수송 목적', '北 매체 괌 미군기지 무수단 미사일 공격 영상 공개', '北 200일전투 독려 가속…평양시 사상일꾼회의']


## 21.4 부정과 문맥

부정 표현은 짧은 단어 하나로 의미를 뒤집습니다. 그런데 그 단어가 **불용어로 지워지거나, 형태소 원형화로 사라지면** 정반대 문장이 같아집니다.

In [10]:
pair("부정", "ko", "실적이 좋다", "실적이 좋지 않다")
print()
pair("부정", "en", "the new card works well", "the new card does not work")

[부정] (ko)
  A: "실적이 좋다" → ['실적', '좋다']
  B: "실적이 좋지 않다" → ['실적', '좋다']
  코사인 유사도 = 1.000  (벡터 완전히 동일)

[부정] (en)
  A: "the new card works well" → ['new', 'card', 'works']
  B: "the new card does not work" → ['new', 'card', 'work']
  코사인 유사도 = 0.693


- 한국어: `좋지`는 원형 `좋다`가 되고, `않다`는 10장에서 정한 **불용어**라 지워집니다. 정반대 문장의 벡터가 완전히 같아집니다.
- 영어: `not`, `does`는 NLTK 불용어, `well`은 10장에서 정한 도메인 불용어라 모두 지워집니다. 남은 차이는 `works`/`work`(어간 추출을 하지 않음)뿐입니다. 어간 추출을 했다면 이 쌍도 완전히 같은 벡터가 됐을 것입니다.

10장에서 "NLTK 목록에 부정어가 들어 있음을 알고 썼다"고 적은 대가가 바로 이것입니다. 주제 분류(우주냐 야구냐)에는 영향이 작지만, **감성 분석이나 사실 확인**처럼 부정이 중요한 작업이라면 치명적입니다.

## 21.5 희소성, 고차원, OOV

### 희소성과 고차원
문서 하나는 수천~수만 차원 중 **극히 일부 칸만** 0이 아닙니다. 짧은 문서일수록 더 그렇습니다.

In [11]:
for lang, df in [("en", en), ("ko", ko)]:
    nnz = df["tokens"].map(lambda t: len({w for w in t if w in VEC[lang].vocabulary_}))
    V = len(VEC[lang].vocabulary_)
    print(f"{lang}: 어휘 {V:,}차원 중 문서당 0이 아닌 칸 평균 {nnz.mean():.1f}개 ({nnz.mean() / V:.3%}), 중앙값 {nnz.median():.0f}개")

rng = np.random.default_rng(RANDOM_STATE)
sample = rng.choice(len(ko), 1000, replace=False)
S = X_ko[sample] @ X_ko[sample].T
off_diag = S[~np.eye(len(sample), dtype=bool)]
print(f"\n한국어 제목 1,000개에서 뽑은 문서 쌍 중 유사도가 정확히 0인 비율: {(off_diag == 0).mean():.1%}")

en: 어휘 13,387차원 중 문서당 0이 아닌 칸 평균 55.9개 (0.418%), 중앙값 31개
ko: 어휘 6,332차원 중 문서당 0이 아닌 칸 평균 7.2개 (0.114%), 중앙값 7개

한국어 제목 1,000개에서 뽑은 문서 쌍 중 유사도가 정확히 0인 비율: 97.1%


한국어 제목은 문서당 0이 아닌 칸이 몇 개뿐이라, **대부분의 문서 쌍이 겹치는 단어가 하나도 없습니다**(유사도 0). 이 상태에서는 단어 하나가 겹치느냐 마느냐에 결과가 크게 좌우됩니다. 20장에서 짧은 문서의 정확도가 낮았던 이유입니다.

### OOV: 학습 때 없던 단어
TF-IDF는 학습한 어휘 밖의 단어를 **버립니다**. 새로 등장한 개념은 아예 보이지 않습니다.

In [12]:
pair("어휘 밖 단어(OOV)", "ko", "코로나 백신", "백신 접종")

dates = pd.read_json(YNAT_PATH)["date"].str[:4]
print(f"\nKLUE-YNAT 기사 연도 범위: {dates.min()}~{dates.max()}")
print("'코로나'가 어휘에 있나?", "코로나" in VEC["ko"].vocabulary_)

[어휘 밖 단어(OOV)] (ko)
  A: "코로나 백신" → ['코로나', '백신']
  B: "백신 접종" → ['백신', '접종']
  코사인 유사도 = 0.000
  어휘 사전에 없어 버려진 단어: ['백신', '접종', '코로나']



KLUE-YNAT 기사 연도 범위: 2016~2019
'코로나'가 어휘에 있나? False


`코로나`뿐 아니라 `백신`, `접종`도 어휘에 없습니다. 이 데이터는 2016~2019년 기사 제목이라 이런 단어가 아예 없거나 `min_df`보다 드물게 나옵니다. 세 단어가 모두 버려져 두 문장이 모두 0 벡터가 되고, 유사도는 0입니다. 데이터가 수집된 시기 이후에 생긴 단어는 색인을 다시 만들기 전까지 검색도 분류도 할 수 없습니다. 16장에서 `코로나 확진자` 검색이 엉뚱한 결과를 낸 이유가 이것입니다.

## 📊 과제 결과 예시 형식으로 정리

과제 결과 예시의 `=== BoW 한계 분석 ===` 형식으로 지금까지의 사례를 출력합니다.

In [13]:
print("=== BoW 한계 분석 ===")
for n, kind in enumerate(dict.fromkeys(r["한계"] for r in RESULTS), start=1):
    print(f"\n[문제 {n}: {kind}]")
    for r in (r for r in RESULTS if r["한계"] == kind):
        print(f'"{r["문장 A"]}" vs "{r["문장 B"]}" ({r["언어"]}) → 코사인 {r["코사인"]:.3f}')

summary = pd.DataFrame(RESULTS)
print("\n" + summary.to_string(index=False))

=== BoW 한계 분석 ===

[문제 1: 어순 무시]
"the dog bit the man" vs "the man bit the dog" (en) → 코사인 1.000
"한국이 일본을 이겼다" vs "일본이 한국을 이겼다" (ko) → 코사인 1.000

[문제 2: 동음이의어]
"배가 고프다" vs "배를 타고 바다로" (ko) → 코사인 0.317
"apple released a new mac" vs "apple pie recipe" (en) → 코사인 0.385

[문제 3: 동의어·표기 차이]
"북한" vs "北" (ko) → 코사인 0.000
"자동차 판매" vs "차량 판매" (ko) → 코사인 0.396
"car" vs "automobile" (en) → 코사인 0.000
"nasa shuttle launch" vs "space agency orbiter liftoff" (en) → 코사인 0.000

[문제 4: 부정]
"실적이 좋다" vs "실적이 좋지 않다" (ko) → 코사인 1.000
"the new card works well" vs "the new card does not work" (en) → 코사인 0.693

[문제 5: 어휘 밖 단어(OOV)]
"코로나 백신" vs "백신 접종" (ko) → 코사인 0.000

          한계 언어                     문장 A                         문장 B   코사인
       어순 무시 en      the dog bit the man          the man bit the dog 1.000
       어순 무시 ko              한국이 일본을 이겼다                  일본이 한국을 이겼다 1.000
       동음이의어 ko                   배가 고프다                    배를 타고 바다로 0.317
       동음이의어 en apple released a new mac    

## 21.6 개선 방향

### n-gram 미리 보기: 부정을 구별할 수 있을까?
**n-gram**은 연속한 n개 단어를 하나의 특성으로 씁니다. bigram(2-gram)이면 `not good`, `좋다_않다`가 하나의 열이 됩니다. sklearn의 `ngram_range`로 미리 확인합니다. 두 문장만으로 어휘를 만드는 단순 비교입니다.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer


def cos_counts(a, b, **kwargs) -> float:
    X = CountVectorizer(token_pattern=r"(?u)\b\w+\b", **kwargs).fit_transform([a, b]).toarray().astype(float)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return float(X[0] @ X[1])


a, b = "this movie is good", "this movie is not good"
print(f'"{a}" vs "{b}"')
for stop in ["english", None]:
    for ngram in [(1, 1), (1, 2)]:
        print(f"  불용어={str(stop):<8} ngram_range={ngram}: 코사인 {cos_counts(a, b, stop_words=stop, ngram_range=ngram):.3f}")

"this movie is good" vs "this movie is not good"
  불용어=english  ngram_range=(1, 1): 코사인 1.000
  불용어=english  ngram_range=(1, 2): 코사인 1.000
  불용어=None     ngram_range=(1, 1): 코사인 0.894
  불용어=None     ngram_range=(1, 2): 코사인 0.756


- 불용어 목록으로 `not`을 지우면 n-gram을 써도 두 문장이 **똑같습니다**(1.000). 지운 단어는 n-gram으로도 되살아나지 않습니다.
- `not`을 남기고 bigram을 쓰면 `not good`이라는 새 특성이 생겨 유사도가 더 내려갑니다.

한국어도 같습니다. `않다`를 불용어에서 빼야 bigram `좋다_않다`가 생깁니다.

In [15]:
def bigrams(tokens):
    return tokens + [f"{x}_{y}" for x, y in zip(tokens, tokens[1:])]


def cos_tokens(ta, tb) -> float:
    vocab = sorted(set(ta) | set(tb))
    va = np.array([ta.count(w) for w in vocab], float)
    vb = np.array([tb.count(w) for w in vocab], float)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


s1, s2 = "실적이 좋다", "실적이 좋지 않다"
for use_stop in [True, False]:
    prep = KoreanPreprocessor("okt", use_stopwords=use_stop)
    t1, t2 = prep(s1), prep(s2)
    print(f"불용어 {'적용' if use_stop else '미적용'}: {t1} / {t2}")
    print(f"  unigram 코사인 {cos_tokens(t1, t2):.3f}, +bigram 코사인 {cos_tokens(bigrams(t1), bigrams(t2)):.3f}")

불용어 적용: ['실적', '좋다'] / ['실적', '좋다']
  unigram 코사인 1.000, +bigram 코사인 1.000
불용어 미적용: ['실적', '좋다'] / ['실적', '좋다', '않다']
  unigram 코사인 0.816, +bigram 코사인 0.775


n-gram은 어순과 부정의 **일부**를 잡지만, 어휘가 폭발적으로 늘어나 희소성은 더 심해집니다. 그래서 실제로는 아래처럼 문제마다 다른 방법을 씁니다.

| 문제 | 개선 방법 | 무엇이 해결되고 무엇이 남나 |
|---|---|---|
| 어순·부정 | n-gram 특성, 부정어를 불용어에서 빼고 다음 단어와 결합 | 짧은 범위의 순서는 잡지만 어휘가 크게 늘어난다 |
| 동의어·표기 차이 | 동의어 사전(북한=北, 美=미국), 쿼리 확장 | 사전에 넣은 쌍만 해결된다 |
| 동의어 전반 | **단어 임베딩**(Word2Vec, FastText): 비슷한 문맥의 단어를 가까운 벡터로 | 단어마다 벡터가 하나라 다의어는 여전히 문제 |
| 다의어·문맥 | **문맥 임베딩**(BERT 계열), 문장 임베딩 검색 | 같은 `배`도 문장에 따라 다른 벡터. 계산 비용이 크다 |
| OOV·신조어 | **서브워드** 토큰화(BPE, WordPiece), 색인 주기적 갱신 | `코로나` 같은 새 단어도 조각으로 표현 |
| 형태소 오류 | 사용자 사전(삼성전자, 5G), 다른 분석기 비교 | 사전 관리 비용 |
| 짧은 문서 | 제목 + 본문 사용, 문자 n-gram, 사전 학습 임베딩 | 정보량 자체를 늘린다 |

**후속 학습 순서 제안**: 단어 임베딩(Word2Vec, FastText) → BM25(TF 포화와 문서 길이 보정) → 문맥 임베딩(BERT)과 벡터 검색. 다음 과제 **A4-2**(정보 추출·감성 분석)에서는 이 장에서 본 **부정 처리** 문제를 규칙으로 직접 다루게 됩니다.

## 📝 리포트용 정리: BoW 한계와 오분류의 연결

리포트 "한계와 개선 방향"에는 위 사례 표(`summary`)와 함께, **20장의 오분류 사례가 어떤 한계에서 나왔는지** 연결해서 씁니다.

| 20장의 오분류 원인 유형 | 해당하는 BoW 한계 | 설명 |
|---|---|---|
| 주제가 겹침 (graphics ↔ windows.x, 야구 ↔ 하키) | 동의어·문맥 부족 | 두 카테고리가 같은 단어를 공유하고, 구별해 줄 문맥은 보지 않는다 |
| 문맥이 필요함 (인용문이 지워진 답글) | 문맥 손실 | 문서 하나를 독립된 단어 주머니로 본다 |
| 다의어에 끌려감 (`space`, `배`) | 동음이의어 | 다른 뜻이 같은 차원에 더해진다 |
| 짧은 문서 | 희소성 | 0이 아닌 칸이 적어 단어 하나에 결과가 좌우된다 |
| 전처리 부산물 (`5G` → `g`, `삼성전자` → `전자`) | 토큰화 한계 + OOV | 단어 단위가 잘못되면 BoW는 복구할 방법이 없다 |
| 레이블 모호 (주석자 불일치) | 모델 밖의 문제 | BoW 지표에 섞여 나오므로 따로 구분해야 한다 |

## ✍️ 실습: 나만의 한계 사례 만들기

`pair()` 함수로 아래 사례를 직접 만들어 보세요.
1. 영어 **어순** 사례 하나 (주어와 목적어가 바뀌는 문장)
2. 한국어 **어순** 사례 하나
3. 한국어 **동의어·표기** 사례 하나 (예: 한자 약어와 한글)

어휘에 없는 단어가 섞이면 결과가 달라지니, 출력의 "버려진 단어"를 확인하면서 고르세요.

In [16]:
# 여기에 작성하세요

**정답 (예시)**

In [17]:
pair("어순 무시", "en", "the yankees beat the red sox", "the red sox beat the yankees")
print()
pair("어순 무시", "ko", "삼성이 애플을 고소했다", "애플이 삼성을 고소했다")
print()
pair("동의어·표기 차이", "ko", "美 대통령", "미국 대통령")

[어순 무시] (en)
  A: "the yankees beat the red sox" → ['yankees', 'beat', 'red', 'sox']
  B: "the red sox beat the yankees" → ['red', 'sox', 'beat', 'yankees']
  코사인 유사도 = 1.000  (벡터 완전히 동일)

[어순 무시] (ko)
  A: "삼성이 애플을 고소했다" → ['삼성', '애플', '고소하다']
  B: "애플이 삼성을 고소했다" → ['애플', '삼성', '고소하다']
  코사인 유사도 = 1.000  (벡터 완전히 동일)
  어휘 사전에 없어 버려진 단어: ['고소하다']

[동의어·표기 차이] (ko)
  A: "美 대통령" → ['美', '대통령']
  B: "미국 대통령" → ['미국', '대통령']
  코사인 유사도 = 0.410


`美 대통령`과 `미국 대통령`은 `대통령`이 겹쳐서 0은 아니지만, `美`와 `미국`은 서로 기여하지 못합니다.

## 🧐 퀴즈

**Q1.** "the dog bit the man"과 "the man bit the dog"의 TF-IDF 벡터가 같은 이유를 BoW의 정의로 설명하세요.
<details><summary>▶ 정답 보기</summary>

BoW는 문서에 어떤 단어가 몇 번 나왔는지만 세고 순서는 버립니다. 두 문장은 같은 단어를 같은 횟수만큼 가지므로 카운트 벡터가 같고, 같은 IDF를 곱하고 정규화해도 같은 벡터가 됩니다.
</details>

**Q2.** 공백 토큰화에서 "고양이가 쥐를 잡았다"와 "쥐가 고양이를 잡았다"의 벡터가 다르면, 공백 토큰화는 어순을 반영한다고 할 수 있나요?
<details><summary>▶ 정답 보기</summary>

아닙니다. 차이는 조사(`가`/`를`)가 토큰에 남아서 생긴 것입니다. 순서만 바꾼 "쥐를 고양이가 잡았다"는 원래 문장과 같은 벡터가 되므로 여전히 어순은 보지 않습니다.
</details>

**Q3.** "북한" 검색으로 `北`만 쓴 기사를 찾지 못하는 이유와 개선 방법 두 가지는?
<details><summary>▶ 정답 보기</summary>

`북한`과 `北`이 서로 다른 차원이라 겹치는 부분이 없어 내적이 0이기 때문입니다. 동의어 사전으로 표기를 통일하거나 쿼리를 확장하는 방법, 또는 비슷한 단어를 가까운 벡터로 만드는 단어 임베딩을 쓰는 방법이 있습니다.
</details>

**Q4.** "실적이 좋다"와 "실적이 좋지 않다"가 같은 벡터가 된 두 가지 원인은? n-gram만 추가하면 해결되나요?
<details><summary>▶ 정답 보기</summary>

`좋지`가 원형 `좋다`로 바뀌고, 부정을 나타내는 `않다`가 불용어로 지워졌기 때문입니다. n-gram만 추가해서는 해결되지 않습니다. 지워진 `않다`는 n-gram에도 나타나지 않으므로, 부정어를 불용어에서 빼고 n-gram을 함께 써야 `좋다_않다` 같은 특성이 생깁니다.
</details>

**Q5.** 단어 임베딩(Word2Vec)이 동의어 문제는 줄여 주지만 다의어 문제는 남기는 이유는?
<details><summary>▶ 정답 보기</summary>

Word2Vec은 비슷한 문맥에 나오는 단어를 가까운 벡터로 만들어 동의어를 가깝게 하지만, 단어마다 벡터가 하나뿐이라 `배`(신체)와 `배`(선박)를 구별하지 못합니다. 문장마다 다른 벡터를 만드는 문맥 임베딩(BERT)이 필요합니다.
</details>

## ✅ 체크포인트
- [ ] 어순, 동음이의어, 동의어, 부정, OOV, 희소성 사례를 실제 코사인 유사도와 함께 말할 수 있다.
- [ ] 각 한계가 "순서를 버린다", "글자가 같으면 같은 차원", "학습한 어휘만 쓴다" 중 어디서 나오는지 설명할 수 있다.
- [ ] 20장의 오분류 원인과 BoW 한계를 연결할 수 있다.
- [ ] 한계마다 개선 방법과 그 방법의 남는 문제를 말할 수 있다.

## 🔗 과제 연결과 다음 장
- 학습 목표 ④ "BoW 방식이 문맥과 어순을 반영하지 못하는 한계를 구체적 사례와 함께 설명"을 이 장의 사례 표로 답할 수 있습니다. 표와 개선 방향은 README "한계와 개선 방향"에 들어갑니다. 프로젝트에서는 `src/bow_limits.py`(`main.py --mode bow`)가 같은 사례를 출력합니다.
- **22장. 통합과 리포트 작성**에서 지금까지의 코드를 모듈로 정리하고, README와 제출 체크리스트를 완성합니다.